# 학습 모니터링

학습 로그를 읽어 진행 상황을 봅니다. 학습 중에도 실행할 수 있고, 셀을 다시 실행하면 최신 상태가 나타납니다.

## 1. 설정 변수

- `CONFIG_NAMES`: 볼 설정 이름들. 여러 개를 적으면 한 그림에 겹쳐 비교합니다. 그림 파일은 첫 번째 실행의 `logs/`에 저장됩니다.
- `SAMPLE_STEP`: 예측 표본을 볼 검증 step. `None`이면 가장 최근 검증입니다.

In [ ]:
CONFIG_NAMES = ["config002"]
SAMPLE_STEP = None

필요한 모듈을 불러옵니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from scripts import config as config_utils
from scripts import datautils, evalutils, logutils

save_dir = config_utils.logs_dir(CONFIG_NAMES[0])

## 2. 실행 이력

학습을 시작하거나 재개할 때마다 한 행이 생깁니다.

In [ ]:
for name in CONFIG_NAMES:
    runs = logutils.read_runs(config_utils.db_path(name))
    print(name if not runs.empty else f"{name}: 로그 없음")
    if not runs.empty:
        display(runs.drop(columns=["config_json", "env_json"]))

## 3. 요약

현재 step, 최근 손실과 검증 지표, 최고 검증 Tanimoto와 그 step입니다.

In [ ]:
display(logutils.summarize(CONFIG_NAMES))

## 4. 손실 곡선

실선은 훈련 손실, 파선은 검증 손실입니다.

In [ ]:
panels = {"loss": [("train", "loss"), ("val", "loss")]}
fig = logutils.plot_curves(CONFIG_NAMES, panels, save_dir / "plot_loss.png")
plt.show()

## 5. 검증 지표 곡선

`tanimoto`는 Kaggle 점수와 같은 정의이고, `valid_ratio`는 유효한 예측의 비율, `exact_match`는 완전 일치율입니다.

In [ ]:
panels = {
    "val tanimoto": [("val", "tanimoto")],
    "val valid_ratio": [("val", "valid_ratio")],
    "val exact_match": [("val", "exact_match")],
}
fig = logutils.plot_curves(CONFIG_NAMES, panels, save_dir / "plot_metrics.png")
plt.show()

## 6. 학습률 곡선

디코더의 학습률입니다.

In [ ]:
panels = {"learning rate": [("train", "lr")]}
fig = logutils.plot_curves(CONFIG_NAMES, panels, save_dir / "plot_lr.png")
plt.show()

## 7. 예측 표본

검증 때마다 기록한 같은 분자들의 정답과 예측을 봅니다. 그림은 왼쪽부터 입력 이미지, 정답 구조, 예측 구조입니다.

In [ ]:
for name in CONFIG_NAMES:
    samples = logutils.read_samples(config_utils.db_path(name), SAMPLE_STEP)
    if samples.empty:
        print(f"{name}: 기록된 예측 표본 없음")
        continue
    print(f"{name}, step {samples['step'].iloc[0]}")
    display(samples[["image_id", "true_smiles", "pred_smiles", "tanimoto", "is_valid"]])

    shown = samples.head(4)
    image_dir = Path(config_utils.load_snapshot(name)["data"]["data_dir"]) / datautils.TRAIN_IMAGE_DIR
    fig = evalutils.plot_structures(
        [image_dir / f"{image_id}.png" for image_id in shown["image_id"]],
        {"true": shown["true_smiles"].tolist(), "prediction": shown["pred_smiles"].tolist()},
    )
    plt.show()